In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

# 1. Initialize Search Client
search_client = SearchClient(
    endpoint="https://rush-lyric-ai-search.search.windows.net",
    index_name="integrated-index",
    credential=AzureKeyCredential("<REDACTED_API_KEY>")
)

# 2. Define the Query
# Hybrid search uses BOTH search_text (keyword) AND vector_queries (semantic)
query_text = "space travel Cygnus"

vector_query = VectorizableTextQuery(
    text=query_text, 
    fields="vector", 
    k_nearest_neighbors=3
)

# 3. Execute Hybrid Search
# By providing both search_text and vector_queries, RRF automatically merges the results.
results = search_client.search(
    search_text=query_text,   # KEYWORD PART
    vector_queries=[vector_query], # SEMANTIC PART
    select=["id", "content"],
    top=3
)

print(f"Hybrid Search Results for: '{query_text}'\n")
for result in results:
    score = result.get('@search.score', 0)
    # Clean up the snippet to avoid f-string backslash errors in Python < 3.12
    content_snippet = result['content'][:150].replace('\n', ' ')
    print(f"Score: {score:.4f} | ID: {result['id']}")
    print(f"Snippet: {content_snippet}...\n")

Hybrid Search Results for: 'space travel Cygnus'

Score: 0.0331 | ID: cygnus_x1_the_voyage
Snippet: Title: Cygnus X-1: The Voyage Album: A Farewell to Kings Lyrics: Prologue: In the constellation of Cygnus, there lurks a mysterious, invisible force: ...

Score: 0.0325 | ID: cygnus_x1_hemispheres
Snippet: Title: Cygnus X-1: Hemispheres Album: Hemispheres Lyrics: Prelude  When our weary world was young The struggle of the ancients first began The gods of...

Score: 0.0306 | ID: virtuality
Snippet: Title: Virtuality Album: Test for Echo Lyrics: Like a shipwrecked mariner adrift on an unknown sea Clinging to the wreckage of the lost ship Fantasy I...

